# 05 — Projecting a drift onto a parameterized map

Now, instead of moving every particle independently, we represent the current distribution as
$$
q_\theta=(T_\theta)_\#\rho_0,
$$
and restrict the drift to motions that can be produced by changing the parameters $\theta$.
We consider two parameterizations:

1. **Affine map**

   $$
   T_\theta(z)
   =
   R(\phi)\operatorname{diag}(s_1,s_2)z+\mu.
   $$
   Its pushforward of a Gaussian latent distribution is exactly Gaussian, and its pullback metric is available in closed form.

2. **One-hidden-layer MLP**

   $$
   T_\theta(z)
   =
   W_2\tanh(W_1z+b_1)+b_2.
   $$
   This map is more flexible, but its pullback metric must be estimated on a fixed latent batch, introducing Monte Carlo estimation error.

We study:
- Euclidean versus pullback-preconditioned updates;
- Spectrum and conditioning of the pullback metric;
- How well a parameterized map tracks the free particle drift;
- The computationnal tradeoff when approximating the pullback metric with $K$ CG descent steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from IPython.display import HTML, display

from msflow import *
from msflow.parametric import (AffineMap, MLPMap, optimize, moment_kl, kl_gaussian,
                               mc_noise_floor, free_drift, transport_project,
                               distance_to_free, W1)

pd.set_option("display.width", 180)
C_E, C_N, C_T, C_F = "#D62828", "#1D3557", "#2A9D8F", "#E9C46A"

---
## 0 · Target and map families

In [ ]:
TARGET_MEAN = (5.0, 3.0)
TARGET_ROTATION = np.pi / 5
TARGET_VARIANCES = (4.0, 0.15)
LATENT_N, LATENT_SEED, VIZ_N = 3000, 42, 400
MLP_H, MLP_SEED, MLP_INIT_SCALE = 8, 123, 0.5
THETA0_AFF = np.array([0.0, np.log(0.4), np.log(0.4), -0.5, -0.5])
OPT_RIDGE, OPT_MAX_STEP, TRACK_COND = 1e-4, 10.0, True

rng = np.random.default_rng(LATENT_SEED)
phi = TARGET_ROTATION
R = np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]])
mu_p = np.asarray(TARGET_MEAN)
S_p = R @ np.diag(TARGET_VARIANCES) @ R.T
kappa_p = np.linalg.cond(S_p)
print(f"target eigenvalues {np.round(np.linalg.eigvalsh(S_p), 4)},  condition number {kappa_p:.1f}")

Z = rng.standard_normal((LATENT_N, 2))
Z_VIZ = Z[:VIZ_N]

affine = AffineMap()
theta0_aff = THETA0_AFF.copy()
mlp = MLPMap(h=MLP_H)
theta0_mlp = mlp.init(np.random.default_rng(MLP_SEED), scale=MLP_INIT_SCALE)
print(f"affine: {affine.dim} parameters      MLP: {mlp.dim} parameters (5h+2, h={mlp.h})")

def plot_ellipse(ax, mu, S, n_std=2.0, **kw):
    vals, vecs = np.linalg.eigh(S)
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse(mu, 2 * n_std * np.sqrt(max(vals[1], 1e-12)),
                         2 * n_std * np.sqrt(max(vals[0], 1e-12)), angle=ang, **kw))

---
## 1 · Sweeping step size before comparing preconditioners

In [ ]:
LRS = np.logspace(-3, 0, 13)
N_STEPS, BAR = 300, 1e-2

def sweep(model, theta0, lrs, **kw):
    out = {}
    for method in ("euclidean", "natural"):
        finals = []
        for lr in lrs:
            tr = optimize(model, theta0, Z, mu_p, S_p, lr=lr, n_steps=N_STEPS,
                          method=method, track_cond=False, **kw)
            v = tr.loss[-1]
            finals.append(max(v, 0.0) if np.isfinite(v) else np.nan)
        out[method] = np.array(finals)
    return out

sw_aff = sweep(affine, theta0_aff, LRS, ridge=OPT_RIDGE, max_step=OPT_MAX_STEP)
sw_mlp = sweep(mlp, theta0_mlp, LRS, ridge=OPT_RIDGE, max_step=OPT_MAX_STEP)

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.3))
for a, sw, ttl in [(ax[0], sw_aff, f"Exp 1 — affine (5 params)"),
                   (ax[1], sw_mlp, f"Exp 2 — MLP ({mlp.dim} params)")]:
    for method, c in [("euclidean", C_E), ("natural", C_N)]:
        a.loglog(LRS, np.maximum(sw[method], 1e-12), "o-", color=c, lw=1.8,
                 label="Euclidean (drifting)" if method == "euclidean" else "natural (pullback)")
    a.set(xlabel=r"learning rate $\eta$", ylabel=r"KL after 300 steps", title=ttl)
    a.legend(fontsize=8); a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "05_lr_sweep")

lr_tab = pd.DataFrame({
    "best KL": [np.nanmin(sw[m]) for sw in (sw_aff, sw_mlp) for m in ("euclidean", "natural")],
    "best lr": [LRS[np.nanargmin(sw[m])] for sw in (sw_aff, sw_mlp)
                for m in ("euclidean", "natural")],
    f"# lr with KL < {BAR:g}": [int(np.nansum(sw[m] < BAR)) for sw in (sw_aff, sw_mlp)
                                for m in ("euclidean", "natural")],
    "lr range that works": [
        (f"{LRS[np.nanargmax(sw[m] < BAR)]:.3g}-{LRS[len(LRS) - 1 - np.nanargmax((sw[m] < BAR)[::-1])]:.3g}"
         if np.nansum(sw[m] < BAR) else "none")
        for sw in (sw_aff, sw_mlp) for m in ("euclidean", "natural")],
}, index=pd.MultiIndex.from_product([["affine", "MLP"], ["euclidean", "natural"]],
                                    names=["map", "method"]))
save_table(lr_tab, "05_lr_sweep",
           caption=rf"Learning-rate sweep: best achievable loss, and how many of the "
                   rf"{len(LRS)} step sizes reach $\mathrm{{KL}} < {BAR:g}$.")
lr_tab

---
## 2 · Trajectories at each method's best sampled step size

In [ ]:
lr_aff = {m: LRS[np.nanargmin(sw_aff[m])] for m in ("euclidean", "natural")}
lr_mlp = {m: LRS[np.nanargmin(sw_mlp[m])] for m in ("euclidean", "natural")}
print("best lr — affine:", lr_aff, "  MLP:", lr_mlp)

tr_aff = {m: optimize(affine, theta0_aff, Z, mu_p, S_p, lr=lr_aff[m],
                      n_steps=N_STEPS, method=m, ridge=OPT_RIDGE, max_step=OPT_MAX_STEP, track_cond=TRACK_COND) for m in lr_aff}
tr_mlp = {m: optimize(mlp, theta0_mlp, Z, mu_p, S_p, lr=lr_mlp[m],
                      n_steps=N_STEPS, method=m, ridge=OPT_RIDGE, max_step=OPT_MAX_STEP, track_cond=TRACK_COND) for m in lr_mlp}

fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.2))
for m, c in [("euclidean", C_E), ("natural", C_N)]:
    ax[0].semilogy(np.maximum(tr_aff[m].loss, 1e-12), color=c, lw=2, label=m)
    ax[1].semilogy(np.maximum(tr_mlp[m].loss, 1e-12), color=c, lw=2, label=m)
ax[0].set(xlabel="iteration", ylabel="KL", title="Exp 1 — affine")
ax[1].set(xlabel="iteration", ylabel="KL", title=f"Exp 2 — MLP ({mlp.dim} params)")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
ax[2].semilogy(tr_aff["natural"].cond_G, color=C_T, lw=2, label="affine")
ax[2].semilogy(np.arange(0, len(tr_mlp["natural"].cond_G)), tr_mlp["natural"].cond_G,
               color=C_N, lw=2, label="MLP")
ax[2].set(xlabel="iteration", ylabel=r"$\kappa(G(\theta_k))$",
          title=r"conditioning of the pullback metric")
ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); save(fig, "05_kl_convergence");

In [ ]:
snaps = [0, 5, 20, 60, 150, N_STEPS]
fig, axes = plt.subplots(4, len(snaps), figsize=(1.45 * len(snaps), 6.8), squeeze=False)
rows = [("affine", affine, tr_aff["euclidean"], "Euclidean", C_E),
        ("affine", affine, tr_aff["natural"], "natural", C_N),
        ("MLP", mlp, tr_mlp["euclidean"], "Euclidean", C_E),
        ("MLP", mlp, tr_mlp["natural"], "natural", C_N)]
for i, (mname, model, tr, label, col) in enumerate(rows):
    for j, k in enumerate(snaps):
        a = axes[i, j]
        plot_ellipse(a, mu_p, S_p, facecolor=C_T, alpha=.12, edgecolor=C_T, lw=1.4, ls="--")
        pts = model.forward(tr.theta[min(k, len(tr.theta) - 1)], Z_VIZ)
        a.scatter(*pts.T, s=2, alpha=.4, color=col)
        a.set_xlim(-4, 10); a.set_ylim(-4, 8); a.set_aspect("equal")
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        if i == 0: a.set_title(f"$k={k}$", fontsize=9)
        if j == 0: a.set_ylabel(f"{mname}\n{label}", fontsize=8)
fig.suptitle(r"transported clouds $T_{\theta_k}(z)$", y=1.0)
fig.tight_layout(); save(fig, "05_clouds");

---
## 3 · Spectrum of the empirical pullback metric

We analyze if the previous results are biased by the error committed when inverting a bad-conditionned matrix using the
$\lambda_{\max}/\sqrt N$ heuristic.

In [ ]:
mid = len(tr_mlp["natural"].theta) // 2
nf_mlp = mc_noise_floor(mlp, tr_mlp["natural"].theta[mid], Z)
nf_aff = mc_noise_floor(affine, tr_aff["natural"].theta[mid], Z)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.semilogy(nf_mlp["eigenvalues"], "o-", color=C_N, ms=4, lw=1.4,
            label=f"MLP ({nf_mlp['n_params']} params)")
ax.semilogy(nf_aff["eigenvalues"], "s-", color=C_T, ms=6, lw=1.6,
            label=f"affine ({nf_aff['n_params']} params)")
ax.axhline(nf_mlp["noise_floor"], color=C_E, ls="--", lw=1.8,
           label=rf"heuristic scale $\lambda_{{\max}}/\sqrt{{N}}$ = {nf_mlp['noise_floor']:.1e}")
ax.set(xlabel="eigenvalue index", ylabel=r"$\lambda_i(G)$",
       title=r"spectrum of the empirical pullback metric at mid-training")
ax.legend(fontsize=8); ax.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "05_metric_spectrum")

spec_tab = pd.DataFrame([
    {"map": "affine", **{k: v for k, v in nf_aff.items() if k != "eigenvalues"}},
    {"map": "MLP", **{k: v for k, v in nf_mlp.items() if k != "eigenvalues"}},
]).set_index("map")
spec_tab["frac below heuristic scale"] = spec_tab["n_below"] / spec_tab["n_params"]
save_table(spec_tab, "05_metric_spectrum",
           caption=r"Conditioning of $G(\theta)$ relative to the descriptive "
                   r"scale $\lambda_{\max}/\sqrt N$; this is not an uncertainty bound.")
spec_tab

---
## 4 · Free drift versus a parameterized projection

In [ ]:
BOARD_N, BOARD_SEED = 200, 11
OUTER_STEPS, OUTER_TAU = 80, 0.3
N_PROJ_INNER, LR_PROJ = 20, 0.05
TRANSPORT_RIDGE, TRANSPORT_MAX_STEP = 1e-4, 5.0
board = load_dataset("checkerboard", n=BOARD_N, seed=BOARD_SEED)
kern_lap = get_kernel("Laplacian", 1.0)
kern_gauss = get_kernel("Gaussian", 1.0)

mlp3 = MLPMap(h=16)
theta3 = mlp3.init(np.random.default_rng(77), scale=0.3)
Zp = np.random.default_rng(42).standard_normal((BOARD_N, 2))
x_init = mlp3.forward(theta3, Zp)

results3 = {}
for kname, kern in (("Laplacian", kern_lap), ("Gaussian", kern_gauss)):
    fld = mean_shift(kern)
    free = free_drift(x_init, board.p, fld, n_steps=OUTER_STEPS, tau=OUTER_TAU)
    runs3 = {"free": {"traj": free}}
    for proj in ("euclidean", "natural"):
        runs3[proj] = transport_project(
        mlp3, theta3, Zp, board.p, fld, n_outer=OUTER_STEPS,
        tau=OUTER_TAU, projection=proj, n_proj=N_PROJ_INNER,
        lr_proj=LR_PROJ, ridge=TRANSPORT_RIDGE,
        max_step=TRANSPORT_MAX_STEP)
    results3[kname] = runs3
    print(f"{kname}: done")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.2))
sample_stride = max(1, OUTER_STEPS // 20)
sample_ks = list(range(0, OUTER_STEPS + 1, sample_stride))
if sample_ks[-1] != OUTER_STEPS:
    sample_ks.append(OUTER_STEPS)
for row, kname in enumerate(results3):
    r = results3[kname]
    a = axes[row, 0]
    for proj, c in [("euclidean", C_E), ("natural", C_N)]:
        a.plot(distance_to_free(r[proj]["traj"], r["free"]["traj"]), color=c, lw=2, label=proj)
    a.set(xlabel="outer iteration $k$", ylabel=r"$d_k$ to the free flow",
          title=f"{kname} kernel — trajectory discrepancy")
    a.legend(fontsize=8); a.grid(alpha=.3)

    a = axes[row, 1]
    for name, c in [("free", C_F), ("euclidean", C_E), ("natural", C_N)]:
        a.plot(sample_ks, [W1(r[name]["traj"][k], board.p.X) for k in sample_ks],
               color=c, lw=2, label=name)
    a.set(xlabel="outer iteration $k$", ylabel=r"$W_1(q^k, p)$",
          title=f"{kname} kernel — sample quality")
    a.legend(fontsize=8); a.grid(alpha=.3)
fig.tight_layout(); save(fig, "05_free_vs_parameterized")

exp3 = pd.DataFrame([
    {"kernel": kname, "projection": proj,
     "final distance to free": distance_to_free(r[proj]["traj"], r["free"]["traj"])[-1],
     "final W1 to data": W1(r[proj]["traj"][-1], board.p.X),
     "free-flow W1": W1(r["free"]["traj"][-1], board.p.X)}
    for kname, r in results3.items() for proj in ("euclidean", "natural")
]).set_index(["kernel", "projection"])
save_table(exp3, "05_free_vs_parameterized",
           caption="Free drift versus parameterized trajectories; discrepancy includes "
                   "projection error and subsequent field divergence.")
exp3.round(4)

In [ ]:
r = results3["Laplacian"]
snaps = sorted(set(int(round(OUTER_STEPS * u)) for u in (0, .06, .19, .38, .62, 1)))
fig, axes = plt.subplots(3, len(snaps), figsize=(1.45 * len(snaps), 5.5), squeeze=False)
for i, (name, c) in enumerate([("free", C_F), ("euclidean", C_E), ("natural", C_N)]):
    for j, k in enumerate(snaps):
        a = axes[i, j]
        a.scatter(*board.p.X.T, s=6, alpha=.28, color=C_T)
        a.scatter(*r[name]["traj"][min(k, len(r[name]["traj"]) - 1)].T,
                  s=8, alpha=.7, color=c)
        a.set_xlim(-5, 5); a.set_ylim(-5, 5); a.set_aspect("equal")
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        if i == 0: a.set_title(f"$k={k}$", fontsize=9)
        if j == 0: a.set_ylabel(name, fontsize=9)
fig.suptitle("Laplacian: free drift vs parameterized trajectories", y=1.0)
fig.tight_layout(); save(fig, "05_exp3_snapshots")

---
## 5 · Fixed budget analysis (CG steps)

In [ ]:
BUDGET = 2000
N_PROJ = [1, 2, 5, 10, 20, 50, 100]
CG_RIDGE, CG_ATOL, CG_RTOL = 1e-4, 1e-10, 0.0
free_ref = free_drift(
    x_init, board.p, mean_shift(kern_lap), n_steps=BUDGET, tau=OUTER_TAU)

exp4 = {}
for npj in N_PROJ:
    n_outer = BUDGET // npj
    exp4[npj] = transport_project(
        mlp3, theta3, Zp, board.p, mean_shift(kern_lap),
        n_outer=n_outer, tau=OUTER_TAU, projection="cg", n_proj=npj,
        ridge=CG_RIDGE, cg_atol=CG_ATOL, cg_rtol=CG_RTOL)
    actual = int(exp4[npj]["cg_iterations"].sum())
    print(f"n_proj={npj:3d}  n_outer={n_outer:4d}  actual_CG={actual:5d}  "
          f"W1={W1(exp4[npj]['traj'][-1], board.p.X):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.4))
cmap = plt.cm.viridis(np.linspace(.15, .95, len(N_PROJ)))
for c, npj in zip(cmap, N_PROJ):
    traj = exp4[npj]["traj"]
    step = max(1, len(traj) // 30)
    ks = list(range(0, len(traj), step))
    ax[0].plot([k * npj for k in ks], [W1(traj[k], board.p.X) for k in ks],
               color=c, lw=1.8, label=f"{npj}")
ax[0].set(xlabel=r"requested inner-CG budget $k n_{\rm proj}$", ylabel=r"$W_1$ to data",
          title=f"fixed requested-inner budget B={BUDGET}")
ax[0].legend(fontsize=7, title=r"$n_{\rm proj}$", ncol=2)

finals = [W1(exp4[n]["traj"][-1], board.p.X) for n in N_PROJ]
ax[1].semilogx(N_PROJ, finals, "o-", color="#264653", lw=2.2, ms=7)
ax[1].axhline(W1(free_ref[-1], board.p.X), color=C_F, ls="--", lw=1.5,
              label="free drift (no parameterization)")
best = N_PROJ[int(np.argmin(finals))]
ax[1].scatter([best], [min(finals)], s=160, facecolor="none", edgecolor=C_E, lw=2,
              label=rf"best sampled: $n_{{\rm proj}}={best}$")
ax[1].set(xlabel=r"$n_{\rm proj}$ (CG cap per outer iteration)",
          ylabel=r"final $W_1$ to data", title="inner depth vs outer descent")
ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "05_cg_interpolation")

exp4_tab = pd.DataFrame({
    "n_outer": [BUDGET // n for n in N_PROJ],
    "requested CG cap": [(BUDGET // n) * n for n in N_PROJ],
    "actual CG iterations": [int(exp4[n]["cg_iterations"].sum()) for n in N_PROJ],
    "nonzero CG info": [int(np.count_nonzero(exp4[n]["cg_info"])) for n in N_PROJ],
    "final W1": finals,
}, index=pd.Index(N_PROJ, name="n_proj"))
save_table(exp4_tab, "05_cg_interpolation",
           caption=(f"CG projection under requested-inner budget {BUDGET}; "
                    "outer drift/Jacobian costs are not included."))
exp4_tab.round(4)